<img src="https://raw.githubusercontent.com/MohammedAly22/qwencleo-asr/main/assets/QwenCleo-ASR-Banner.png" width="100%"/>

# 🎙️ QwenCleo-ASR — vLLM Serving & True Streaming

Egyptian Arabic & code-switching speech recognition, built on Qwen3-ASR.

[Model](https://huggingface.co/mohammedaly22/QwenCleo-ASR) · [GitHub](https://github.com/MohammedAly22/qwencleo-asr) · [PyPI](https://pypi.org/project/qwencleo-asr/)

> **Runtime → Change runtime type → GPU** before running. Then run the cells top to bottom.

QwenCleo inherits Qwen3-ASR's **token-by-token streaming** via vLLM.

> ⚠️ **Use the `cu129` nightly.** vLLM publishes the Qwen3-ASR nightly for **cu129** only — this is the exact install from the Qwen3-ASR card. `cu129` is forward-compatible and runs on Colab's CUDA-12.x GPUs. The bare `/nightly` index serves a **CUDA-13** wheel that dies with `ImportError: libcudart.so.13` — avoid it.

> No vLLM at all? The [FastAPI server](https://colab.research.google.com/github/MohammedAly22/qwencleo-asr/blob/main/examples/03_fastapi_server.ipynb) and [Quick Start](https://colab.research.google.com/github/MohammedAly22/qwencleo-asr/blob/main/examples/01_quickstart.ipynb) notebooks give the same transcriptions (without token-by-token streaming).

## 1. Install the vLLM nightly (cu129)

vLLM only publishes the Qwen3-ASR nightly for **cu129** — this is the exact command from the Qwen3-ASR card. `cu129` (CUDA 12.9 runtime libs) is **forward-compatible** and runs on Colab's CUDA 12.x GPUs. Do **not** use the bare `/nightly` index — it serves a CUDA-13 wheel that fails with `libcudart.so.13`.

In [ ]:
!pip install -q uv
import subprocess
subprocess.run(['uv','pip','install','--system','-q','-U','vllm','--pre',
                '--extra-index-url','https://wheels.vllm.ai/nightly/cu129',
                '--extra-index-url','https://download.pytorch.org/whl/cu129',
                '--index-strategy','unsafe-best-match'], check=True)
subprocess.run(['uv','pip','install','--system','-q','vllm[audio]',
                'openai','httpx'], check=True)
!pip install -q qwencleo-asr --no-deps

## 1b. Verify vLLM imports in a FRESH process

The notebook kernel can cache a half-loaded vllm, so we check in a subprocess — exactly what `vllm serve` will do.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, '-c', 'import vllm._C; print("ok")'],
                   capture_output=True, text=True)
if 'ok' in r.stdout:
    print('✅ vLLM C-extension loads')
else:
    print(r.stderr[-1500:])
    raise SystemExit('❌ vLLM failed to import — see error above.')

## 2. Download the model first

Pull weights now so the server boots fast in the next cell.

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('mohammedaly22/QwenCleo-ASR')
print('model downloaded')

## 3. Start a vLLM server in the background

> 🖥️ **Use an Ampere-or-newer GPU (L4 / A100 / H100).** Qwen3-ASR's vLLM path uses **FlashInfer**, which doesn't run on the free Colab **T4 (sm_75)** — it crashes with `BatchPrefillWithPagedKVCache ... invalid argument`. In Colab pick **Runtime → Change runtime type → L4**. On a T4, use the FastAPI / Quick Start notebook instead.

Key flags (matching a verified working setup):
- `VLLM_USE_FLASHINFER_SAMPLER=0` — the FlashInfer sampler JIT-compiles a kernel needing `nvcc`, which Colab lacks; this uses the native sampler instead.
- `--gpu-memory-utilization 0.8` — leave headroom if the GPU is shared.
Logs go to `vllm.log`.

In [ ]:
import subprocess, time, requests, os

env = dict(os.environ, VLLM_USE_FLASHINFER_SAMPLER='0')
cmd = ['vllm', 'serve', 'mohammedaly22/QwenCleo-ASR',
       '--gpu-memory-utilization', '0.8',
       '--max-model-len', '4096',
       '--dtype', 'bfloat16']   # use float16 on older (non-Ampere) GPUs
logf = open('vllm.log', 'w')
proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env)
print('starting vLLM server (logging to vllm.log)...')

up = False
for _ in range(120):          # up to ~10 min
    try:
        requests.get('http://localhost:8000/v1/models', timeout=2)
        up = True; print('✅ server up'); break
    except Exception:
        if proc.poll() is not None:
            print('❌ vLLM exited — last 40 log lines:\n')
            print(''.join(open('vllm.log').readlines()[-40:]))
            raise RuntimeError('vLLM failed to start (see log above)')
        time.sleep(5)
if not up:
    print(''.join(open('vllm.log').readlines()[-40:]))
    raise TimeoutError('vLLM did not come up in time (see log above)')

## 4. Sample audio

In [ ]:
# Grab a sample Egyptian/code-switching clip to transcribe
import urllib.request, os
URL = 'https://huggingface.co/mohammedaly22/QwenCleo-ASR/resolve/main/assets/sample.wav'
FALLBACK = 'https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav'
path = 'sample.wav'
try:
    urllib.request.urlretrieve(URL, path)
except Exception:
    urllib.request.urlretrieve(FALLBACK, path)
print('saved', path, os.path.getsize(path), 'bytes')

## 5. TRUE streaming — `asr.stream()` yields tokens as they arrive

The simplest interface: build the model wrapper and stream from the running server by port. The `language X<asr_text>` prefix is stripped for you.

In [ ]:
from qwencleo_asr import QwenCleoASR

asr = QwenCleoASR()
for delta in asr.stream('sample.wav', port=8000):
    print(delta, end='', flush=True)
print()

## 6. Or use the helpers directly (no model object)

In [ ]:
from qwencleo_asr import stream_vllm, transcribe_vllm

# streaming
for delta in stream_vllm('sample.wav', port=8000, language='Arabic'):
    print(delta, end='', flush=True)
print()

# one-shot
print(transcribe_vllm('sample.wav', port=8000))

## 7. Stop the server

In [ ]:
proc.terminate()